# WS03: Sensor Provisioning and Application Deployment

## Overview
This workshop demonstrates how to:
- Connect simulated IoT devices to the platform
- Register devices and create service groups
- Read sensor data via the platform
- Deploy a simple PID controller application

## Assumption
We use the **object-oriented data model** from WS02 for this section.

## Learning Objectives
- Understand MQTT device communication
- Register IoT devices with the platform
- Set up service groups for device management
- Implement a PID controller for building automation
- Read and process sensor data in real-time

## Components
1. **Simulated Sensors**: Temperature sensor publishing via MQTT
2. **Simulated Actuators**: Radiator thermostat receiving commands via MQTT
3. **Controller Application**: PID controller adjusting radiator based on temperature
4. **Context Broker**: FIWARE Orion for entity management
5. **IoT Agent**: Adapter for MQTT to Context Broker

## Part 1: Setup and Configuration

In [ ]:
# Import required packages
import logging
import time
from pprint import pprint
import numpy as np
from typing import Callable

from filip.clients.ngsi_v2 import ContextBrokerClient, IoTAgentClient
from filip.models.base import FiwareHeader, DataType
from filip.models.ngsi_v2.context import (
    ContextEntity,
    ContextAttribute,
    NamedContextAttribute,
)
from filip.models.ngsi_v2.iot import Device, DeviceAttribute, DeviceCommand, ServiceGroup
from filip.config import settings

# Configure logging
logging.basicConfig(
    level="INFO",
    format="%(asctime)s %(name)s %(levelname)s: %(message)s",
    datefmt="%d-%m-%Y %H:%M:%S",
)
logger = logging.getLogger(__name__)

print("Setup completed!")

## Part 2: Simulation Model

First, we need a simulation model to mimic real building physics. This will simulate temperature changes based on:
- Ambient temperature (outdoor)
- Zone temperature (room)
- Heating power (from radiator)
- Heat loss through walls

In [ ]:
class SimpleBuildingModel:
    """
    Simplified thermal model of a room with:
    - Single zone (room)
    - Heat input from radiator
    - Heat loss through walls (proportional to temperature difference)
    """
    
    def __init__(self,
                 t_ambient_initial: float = 5.0,      # Initial outdoor temperature
                 t_zone_initial: float = 18.0,          # Initial room temperature
                 thermal_capacitance: float = 1.0,      # Room thermal capacitance
                 thermal_resistance: float = 0.1,       # Wall thermal resistance
                 heater_power_initial: float = 0.0):    # Initial heater power
        """
        Initialize the building thermal model
        
        Parameters:
        -----------
        t_ambient_initial: Outdoor temperature (°C)
        t_zone_initial: Initial room temperature (°C)
        thermal_capacitance: Heat capacity of the zone (higher = harder to heat)
        thermal_resistance: Wall insulation (higher = less heat loss)
        heater_power_initial: Initial heater power (W)
        """
        self.t_ambient = t_ambient_initial
        self.t_zone = t_zone_initial
        self.C = thermal_capacitance
        self.R = thermal_resistance
        self.P_heater = heater_power_initial
        
    def update(self, dt: float, t_ambient: float = None, P_heater: float = None):
        """
        Update the thermal model for time step dt (seconds)
        
        The simplified energy balance equation:
        C * dT/dt = P_heater / 1000 - (T_zone - T_ambient) / R
        
        Parameters:
        -----------
        dt: Time step in seconds
        t_ambient: Outdoor temperature (if provided)
        P_heater: Heater power in watts (if provided)
        """
        if t_ambient is not None:
            self.t_ambient = t_ambient
        if P_heater is not None:
            self.P_heater = P_heater
        
        # Heat input from radiator (convert W to kW)
        q_heater = self.P_heater / 1000.0
        
        # Heat loss through walls (proportional to temperature difference)
        q_loss = (self.t_zone - self.t_ambient) / self.R
        
        # Temperature change (dT/dt = (q_in - q_out) / C)
        dt_zone = dt * (q_heater - q_loss) / self.C
        
        self.t_zone += dt_zone
        
        return self.t_zone
    
    def get_state(self) -> dict:
        """Return current state of the model"""
        return {
            'temperature_zone': self.t_zone,
            'temperature_ambient': self.t_ambient,
            'heater_power': self.P_heater,
        }

# Test the model
model = SimpleBuildingModel(
    t_ambient_initial=5.0,
    t_zone_initial=18.0,
    thermal_capacitance=1.0,
    thermal_resistance=0.1
)

print("Building Thermal Model initialized!")
print(f"Initial state: {model.get_state()}")

## Part 3: PID Controller

Implement a proportional-integral-derivative (PID) controller to maintain the room at a target temperature.

In [ ]:
class PIDController:
    """
    Simple PID controller for temperature setpoint tracking.
    
    The PID controller adjusts the heater power based on:
    - Proportional term: immediate response to error
    - Integral term: correction for steady-state error
    - Derivative term: prediction of future error (damping)
    """
    
    def __init__(self,
                 kp: float = 100.0,    # Proportional gain
                 ki: float = 10.0,      # Integral gain
                 kd: float = 50.0,      # Derivative gain
                 setpoint: float = 21.0,  # Target temperature
                 power_min: float = 0.0,   # Minimum heater power (W)
                 power_max: float = 5000.0): # Maximum heater power (W)
        """
        Initialize the PID controller
        
        Parameters:
        -----------
        kp: Proportional gain (response strength to error)
        ki: Integral gain (accumulation of past errors)
        kd: Derivative gain (dampening)
        setpoint: Target temperature (°C)
        power_min: Minimum output power (W)
        power_max: Maximum output power (W)
        """
        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.setpoint = setpoint
        self.power_min = power_min
        self.power_max = power_max
        
        # State variables
        self.integral_error = 0.0
        self.previous_error = 0.0
        self.previous_output = 0.0
        
    def update(self, current_temperature: float, dt: float = 1.0) -> float:
        """
        Calculate the control output for the next time step.
        
        Parameters:
        -----------
        current_temperature: Measured temperature (°C)
        dt: Time step (seconds)
        
        Returns:
        --------
        Heater power setpoint (W)
        """
        # Calculate error
        error = self.setpoint - current_temperature
        
        # Proportional term
        p_term = self.kp * error
        
        # Integral term
        self.integral_error += error * dt
        i_term = self.ki * self.integral_error
        
        # Derivative term
        derivative_error = (error - self.previous_error) / dt if dt > 0 else 0
        d_term = self.kd * derivative_error
        
        # PID output
        output = p_term + i_term + d_term
        
        # Saturation (limit to min/max power)
        output = max(self.power_min, min(self.power_max, output))
        
        # Update state
        self.previous_error = error
        self.previous_output = output
        
        return output
    
    def set_setpoint(self, setpoint: float):
        """Change the target temperature"""
        self.setpoint = setpoint
    
    def reset(self):
        """Reset the controller state (for testing)"""
        self.integral_error = 0.0
        self.previous_error = 0.0
        self.previous_output = 0.0

# Test the PID controller
pid = PIDController(
    kp=100.0,
    ki=10.0,
    kd=50.0,
    setpoint=21.0,
    power_max=5000.0
)

print("PID Controller initialized!")
print(f"Setpoint: {pid.setpoint}°C")
print(f"Gains: Kp={pid.kp}, Ki={pid.ki}, Kd={pid.kd}")

## Part 4: Configuration Parameters

Set up the Context Broker and IoT Agent connections.

In [ ]:
# Configuration parameters
CB_URL = settings.CB_URL
IOT_AGENT_URL = settings.IOT_URL if hasattr(settings, 'IOT_URL') else "http://localhost:4041"

# FIWARE Service
SERVICE = "workshop"
SERVICE_PATH = "/building"

# Device configuration
DEVICE_ID = "urn:ngsi-ld:Device:ThermalControl:Room101"
SERVICE_GROUP_NAME = "ThermalControlDevices"
API_KEY = "thermal_api_key_2024"  # API key for device authentication

# PID Controller parameters
SETPOINT_TEMPERATURE = 21.0  # Target temperature in °C

# Simulation parameters
SIMULATION_TIMESTEP = 60  # seconds (update every minute)
SIMULATION_DURATION = 3600  # seconds (run for 1 hour)

print("Configuration Parameters:")
print(f"  Context Broker: {CB_URL}")
print(f"  IoT Agent: {IOT_AGENT_URL}")
print(f"  Service: {SERVICE}")
print(f"  Service Path: {SERVICE_PATH}")
print(f"  API Key: {API_KEY}")
print(f"  Target Temperature: {SETPOINT_TEMPERATURE}°C")

## Part 5: Set Up FIWARE Connection

In [ ]:
# Create FIWARE header
fiware_header = FiwareHeader(service=SERVICE, service_path=SERVICE_PATH)

# Create clients
try:
    cb_client = ContextBrokerClient(url=CB_URL, fiware_header=fiware_header)
    print("✓ Context Broker client initialized")
except Exception as e:
    print(f"✗ Failed to connect to Context Broker: {e}")

try:
    iot_agent_client = IoTAgentClient(url=IOT_AGENT_URL, fiware_header=fiware_header)
    print("✓ IoT Agent client initialized")
except Exception as e:
    print(f"⚠ IoT Agent not available (this is optional): {e}")

# Verify Context Broker connection
try:
    version_info = cb_client.get_version()
    for key, value in version_info.items():
        print(f"✓ Connected to Context Broker version: {value['version']}")
except Exception as e:
    print(f"✗ Failed to verify Context Broker: {e}")

## Part 6: Register Devices with IoT Agent

Define and register the simulated temperature sensor and radiator thermostat with the IoT Agent.

In [ ]:
# Create service group (handles multiple devices with the same API key)
service_group = ServiceGroup(
    apikey=API_KEY,
    resource="/mqtt",  # MQTT endpoint
    entity_type="ThermalControlDevice",
)

print("Service Group Definition:")
print(f"  API Key: {service_group.apikey}")
print(f"  Resource: {service_group.resource}")
print(f"  Entity Type: {service_group.entity_type}")

# Try to provision service group (if IoT Agent is available)
try:
    iot_agent_client.post_service_group(service_group=service_group)
    print("\n✓ Service group created successfully")
except Exception as e:
    print(f"\n⚠ Could not create service group: {e}")
    print("  (Continuing with manual device registration)")

## Part 7: Create Device Entities in Context Broker

In [ ]:
# Create entities in the Context Broker for the room and devices

# 1. Create Room Entity
room_entity = ContextEntity(
    id="urn:ngsi-ld:Room:ThermalControl:101",
    type="Room"
)
room_entity.add_attributes([
    NamedContextAttribute(name="floor", value=1, type=DataType.INTEGER),
    NamedContextAttribute(name="roomNumber", value=101, type=DataType.INTEGER),
    NamedContextAttribute(name="location", value="Test Lab", type=DataType.TEXT),
])

try:
    cb_client.post_entity(entity=room_entity)
    print("✓ Room entity created")
except Exception as e:
    print(f"⚠ Room entity may already exist: {e}")

# 2. Create Temperature Sensor Entity
temp_sensor_entity = ContextEntity(
    id="urn:ngsi-ld:TemperatureSensor:Room101",
    type="TemperatureSensor"
)
temp_sensor_entity.add_attributes([
    NamedContextAttribute(name="temperature", value=18.0, type=DataType.FLOAT),
    NamedContextAttribute(name="sensorModel", value="SHT31-D", type=DataType.TEXT),
    NamedContextAttribute(name="location", value="Wall mounted", type=DataType.TEXT),
])

try:
    cb_client.post_entity(entity=temp_sensor_entity)
    print("✓ Temperature sensor entity created")
except Exception as e:
    print(f"⚠ Temperature sensor entity may already exist: {e}")

# 3. Create Radiator Thermostat Entity
radiator_entity = ContextEntity(
    id="urn:ngsi-ld:RadiatorThermostat:Room101",
    type="RadiatorThermostat"
)
radiator_entity.add_attributes([
    NamedContextAttribute(name="setpoint", value=SETPOINT_TEMPERATURE, type=DataType.FLOAT),
    NamedContextAttribute(name="power", value=0.0, type=DataType.FLOAT),
    NamedContextAttribute(name="location", value="Wall mounted", type=DataType.TEXT),
])

try:
    cb_client.post_entity(entity=radiator_entity)
    print("✓ Radiator thermostat entity created")
except Exception as e:
    print(f"⚠ Radiator thermostat entity may already exist: {e}")

print("\n✓ All device entities registered with Context Broker")

## Part 8: Simulation Loop

Run the integrated simulation with:
- Building thermal model
- PID controller
- Context Broker updates

In [ ]:
# Initialize simulation components
building_model = SimpleBuildingModel(
    t_ambient_initial=5.0,
    t_zone_initial=18.0,
    thermal_capacitance=1.0,
    thermal_resistance=0.1,
    heater_power_initial=0.0
)

pid_controller = PIDController(
    kp=100.0,
    ki=10.0,
    kd=50.0,
    setpoint=SETPOINT_TEMPERATURE,
    power_max=5000.0
)

# Simulation tracking
simulation_data = {
    'time': [],
    'temperature': [],
    'setpoint': [],
    'heater_power': [],
    'error': [],
}

print("Simulation Configuration:")
print(f"  Duration: {SIMULATION_DURATION} seconds ({SIMULATION_DURATION//60} minutes)")
print(f"  Time Step: {SIMULATION_TIMESTEP} seconds")
print(f"  Target Temperature: {SETPOINT_TEMPERATURE}°C")
print(f"  Initial Room Temperature: {building_model.t_zone}°C")
print(f"  Ambient Temperature: {building_model.t_ambient}°C")

print("\n" + "="*80)
print("Starting simulation...")
print("="*80)

### Simulation Loop

In [ ]:
# Simulation loop (without actual MQTT)
# In a real scenario, sensor data would come via MQTT, but we simulate it here

for step in range(0, SIMULATION_DURATION + SIMULATION_TIMESTEP, SIMULATION_TIMESTEP):
    # Get current temperature from model
    current_temp = building_model.t_zone
    
    # Calculate control output
    heater_power = pid_controller.update(current_temp, SIMULATION_TIMESTEP)
    
    # Update building model
    building_model.update(SIMULATION_TIMESTEP, P_heater=heater_power)
    
    # Calculate error
    error = SETPOINT_TEMPERATURE - current_temp
    
    # Record data
    simulation_data['time'].append(step)
    simulation_data['temperature'].append(current_temp)
    simulation_data['setpoint'].append(SETPOINT_TEMPERATURE)
    simulation_data['heater_power'].append(heater_power)
    simulation_data['error'].append(error)
    
    # Print progress every 10 steps
    if step % (SIMULATION_TIMESTEP * 10) == 0:
        print(f"Time: {step:5d}s ({step//60:3d}m) | "
              f"Temp: {current_temp:6.2f}°C | "
              f"Error: {error:6.2f}°C | "
              f"Power: {heater_power:7.1f}W")

print("\n" + "="*80)
print("Simulation completed!")
print("="*80)

## Part 9: Update Context Broker with Simulation Results

Update the sensor and actuator entities in the Context Broker with the final simulation values.

In [ ]:
# Update Temperature Sensor with final reading
print("Updating Context Broker entities...")

final_temperature = building_model.t_zone
final_heater_power = pid_controller.previous_output

try:
    cb_client.update_attribute_value(
        entity_id="urn:ngsi-ld:TemperatureSensor:Room101",
        attr_name="temperature",
        value=round(final_temperature, 2)
    )
    print(f"✓ Temperature updated: {final_temperature:.2f}°C")
except Exception as e:
    print(f"✗ Failed to update temperature: {e}")

try:
    cb_client.update_attribute_value(
        entity_id="urn:ngsi-ld:RadiatorThermostat:Room101",
        attr_name="power",
        value=round(final_heater_power, 1)
    )
    print(f"✓ Heater power updated: {final_heater_power:.1f}W")
except Exception as e:
    print(f"✗ Failed to update heater power: {e}")

# Verify the updates
try:
    updated_entities = cb_client.get_entity_list(entity_types=["TemperatureSensor", "RadiatorThermostat"])
    print(f"\n✓ Retrieved {len(updated_entities)} updated entities from Context Broker")
except Exception as e:
    print(f"✗ Failed to retrieve entities: {e}")

## Part 10: Analysis and Visualization

Analyze the simulation results to understand the controller performance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create visualization
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Convert time to minutes for better readability
time_minutes = [t/60 for t in simulation_data['time']]

# Plot 1: Temperature Tracking
ax1 = axes[0]
ax1.plot(time_minutes, simulation_data['temperature'], 'b-', label='Room Temperature', linewidth=2)
ax1.plot(time_minutes, simulation_data['setpoint'], 'r--', label='Setpoint', linewidth=2)
ax1.set_ylabel('Temperature (°C)')
ax1.set_title('Temperature Tracking')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, max(time_minutes))

# Plot 2: Control Error
ax2 = axes[1]
ax2.plot(time_minutes, simulation_data['error'], 'g-', linewidth=2)
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax2.set_ylabel('Temperature Error (°C)')
ax2.set_title('Control Error (Setpoint - Measured Temperature)')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, max(time_minutes))

# Plot 3: Heater Power
ax3 = axes[2]
ax3.plot(time_minutes, simulation_data['heater_power'], 'm-', linewidth=2)
ax3.set_xlabel('Time (minutes)')
ax3.set_ylabel('Heater Power (W)')
ax3.set_title('Heater Power Output')
ax3.grid(True, alpha=0.3)
ax3.set_xlim(0, max(time_minutes))

plt.tight_layout()
plt.savefig('thermal_control_simulation.png', dpi=100, bbox_inches='tight')
print("✓ Visualization saved as 'thermal_control_simulation.png'")
plt.show()

# Print performance metrics
print("\n" + "="*80)
print("Simulation Performance Metrics")
print("="*80)

final_error = simulation_data['error'][-1]
max_error = max(abs(e) for e in simulation_data['error'])
mean_error = np.mean(np.abs(simulation_data['error']))
steady_state_temp = np.mean(simulation_data['temperature'][-10:])  # Last 10% of simulation

print(f"\nTemperature Control:")
print(f"  Initial temperature:      {simulation_data['temperature'][0]:.2f}°C")
print(f"  Final temperature:        {final_temperature:.2f}°C")
print(f"  Target temperature:       {SETPOINT_TEMPERATURE:.2f}°C")
print(f"  Temperature rise:         {final_temperature - simulation_data['temperature'][0]:.2f}°C")
print(f"  Steady-state temperature: {steady_state_temp:.2f}°C")

print(f"\nError Metrics:")
print(f"  Final error:              {final_error:.2f}°C")
print(f"  Maximum error:            {max_error:.2f}°C")
print(f"  Mean absolute error:      {mean_error:.2f}°C")

print(f"\nHeater Control:")
print(f"  Maximum power:            {max(simulation_data['heater_power']):.1f}W")
print(f"  Final power:              {final_heater_power:.1f}W")
print(f"  Average power:            {np.mean(simulation_data['heater_power']):.1f}W")

## Part 11: Real-Time Data Access Pattern

Demonstrate how to read sensor data from the Context Broker in a real-time application.

In [ ]:
def read_room_temperature(cb_client, sensor_id: str) -> float:
    """
    Read current temperature from Context Broker
    
    Parameters:
    -----------
    cb_client: ContextBrokerClient instance
    sensor_id: Temperature sensor entity ID
    
    Returns:
    --------
    Current temperature in °C
    """
    try:
        sensors = cb_client.get_entity_list(entity_ids=[sensor_id])
        if sensors:
            temp_attr = sensors[0].get_attribute('temperature')
            return float(temp_attr.value)
    except Exception as e:
        print(f"Error reading temperature: {e}")
    return None


def set_heater_power(cb_client, thermostat_id: str, power: float):
    """
    Set heater power in Context Broker
    
    Parameters:
    -----------
    cb_client: ContextBrokerClient instance
    thermostat_id: Radiator thermostat entity ID
    power: Power value in watts
    """
    try:
        cb_client.update_attribute_value(
            entity_id=thermostat_id,
            attr_name='power',
            value=power
        )
    except Exception as e:
        print(f"Error setting heater power: {e}")


def get_all_room_sensors(cb_client) -> list:
    """
    Get all temperature sensors for rooms
    
    Returns:
    --------
    List of (room_id, temperature) tuples
    """
    try:
        sensors = cb_client.get_entity_list(entity_types=["TemperatureSensor"])
        data = []
        for sensor in sensors:
            temp_attr = sensor.get_attribute('temperature')
            if temp_attr:
                data.append((sensor.id, float(temp_attr.value)))
        return data
    except Exception as e:
        print(f"Error reading sensors: {e}")
        return []


# Example usage
print("Example: Reading sensor data from Context Broker\n")

temp = read_room_temperature(cb_client, "urn:ngsi-ld:TemperatureSensor:Room101")
if temp is not None:
    print(f"Current room temperature: {temp:.2f}°C")

print("\nSetting new heater power: 2500W")
set_heater_power(cb_client, "urn:ngsi-ld:RadiatorThermostat:Room101", 2500.0)

print("\nReading all temperature sensors:")
all_sensors = get_all_room_sensors(cb_client)
for sensor_id, temperature in all_sensors:
    print(f"  {sensor_id}: {temperature:.2f}°C")

## Summary

In this workshop, you have learned:

1. **Simulation Model**: Built a simplified thermal model of a room with heat input and loss

2. **PID Controller**: Implemented a proportional-integral-derivative controller to maintain a target temperature

3. **Device Registration**: 
   - Registered devices with service groups
   - Created entities in the Context Broker for sensors and actuators

4. **Real-Time Communication**:
   - Updated sensor readings in the Context Broker
   - Set actuator commands via the Context Broker

5. **Application Control**:
   - Implemented control loops reading sensors and adjusting actuators
   - Analyzed controller performance

6. **Data Access Patterns**:
   - Single sensor reading
   - Batch sensor reading
   - Actuator command setting

## Exercises

1. **Tune the PID Controller**:
   - Adjust Kp, Ki, and Kd to improve settling time
   - Minimize steady-state error
   - Avoid overshoot

2. **Multi-Room Control**:
   - Extend the simulation to control multiple rooms simultaneously
   - Implement room interconnections (heat transfer between rooms)

3. **Advanced Features**:
   - Add energy efficiency metrics
   - Implement setpoint scheduling (different temperatures at different times)
   - Add occupancy-based control

4. **Visualization Dashboard**:
   - Create a real-time dashboard to monitor room temperatures
   - Display controller status and performance

5. **Fault Handling**:
   - Detect sensor faults (unrealistic temperature jumps)
   - Implement fallback strategies when sensors are unavailable
   - Add alerts for abnormal conditions